# Correlation Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadtalhaishtiaq/ai-orchestrator/blob/main/02-exploratory-data-analysis/04_correlation_analysis.ipynb)

## Learning Objectives
- Understand different types of correlation (Pearson, Spearman, Kendall)
- Master correlation matrix interpretation
- Learn to detect multicollinearity
- Identify feature importance through correlation

---

## 1. What is Correlation?

**Correlation** = Measures the **strength and direction** of the relationship between two variables

**Key Properties:**
- 📊 Value ranges from **-1 to +1**
- ➡️ **Positive (+)**: Both variables increase together
- ⬅️ **Negative (-)**: One increases, other decreases
- 🔵 **Zero (0)**: No linear relationship

**Correlation Strength Guide:**
```
|r| = 0.00 - 0.19  →  Very weak
|r| = 0.20 - 0.39  →  Weak
|r| = 0.40 - 0.59  →  Moderate
|r| = 0.60 - 0.79  →  Strong
|r| = 0.80 - 1.00  →  Very strong
```

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, kendalltau
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

## 2. Dataset: Housing Prices

In [ ]:
# Create synthetic housing dataset
np.random.seed(42)
n = 600

# Base features
area = np.random.normal(1800, 500, n).clip(800, 4000)
bedrooms = (area / 500).astype(int).clip(1, 6)
bathrooms = (bedrooms * 0.75 + np.random.normal(0, 0.5, n)).clip(1, 5).round(1)
age = np.random.exponential(15, n).clip(0, 60).astype(int)
distance_cbd = np.random.gamma(5, 2, n)  # km from city center
garage_spaces = np.random.choice([0, 1, 2, 3], n, p=[0.1, 0.3, 0.5, 0.1])
crime_rate = np.random.exponential(3, n).clip(0, 20).round(1)

# Pollution increases with distance from CBD (non-linear)
pollution_index = (np.log(distance_cbd + 1) * 10 + np.random.normal(0, 3, n)).clip(0, 50).round(1)

# Price depends on multiple factors
price = (
    area * 250 +
    bedrooms * 50000 +
    bathrooms * 40000 -
    age * 3000 -
    distance_cbd * 12000 +
    garage_spaces * 30000 -
    crime_rate * 8000 -
    pollution_index * 2000 +
    np.random.normal(0, 50000, n)
).clip(100000, 2000000)

df = pd.DataFrame({
    'price': price.round(0),
    'area_sqft': area.round(0),
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'age_years': age,
    'distance_cbd_km': distance_cbd.round(1),
    'garage_spaces': garage_spaces,
    'crime_rate': crime_rate,
    'pollution_index': pollution_index,
    'school_rating': np.random.randint(1, 11, n)  # 1-10 scale
})

print("✅ Housing dataset created!")
print(f"Shape: {df.shape}")
df.head(10)

## 3. Types of Correlation

### 3.1 Pearson Correlation (Linear Relationships)

In [ ]:
# Pearson correlation: Measures LINEAR relationship
# Assumptions: Both variables are continuous and approximately normal

pearson_corr, pearson_p = stats.pearsonr(df['area_sqft'], df['price'])

print("📊 Pearson Correlation: Area vs Price")
print(f"Correlation coefficient: {pearson_corr:.3f}")
print(f"P-value: {pearson_p:.4f}")

if pearson_p < 0.05:
    print("✅ Statistically significant (p < 0.05)")
else:
    print("❌ Not statistically significant (p >= 0.05)")

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(df['area_sqft'], df['price'], alpha=0.5, s=30)
plt.xlabel('Area (sqft)', fontsize=12)
plt.ylabel('Price ($)', fontsize=12)
plt.title(f'Area vs Price\nPearson r = {pearson_corr:.3f}', fontsize=14, fontweight='bold')

# Add regression line
z = np.polyfit(df['area_sqft'], df['price'], 1)
p = np.poly1d(z)
plt.plot(df['area_sqft'], p(df['area_sqft']), "r--", linewidth=2, label='Linear fit')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Spearman Correlation (Monotonic Relationships)

In [ ]:
# Spearman correlation: Based on RANKS, measures monotonic relationship
# Better for: Non-linear but monotonic relationships, ordinal data, outliers

spearman_corr, spearman_p = spearmanr(df['distance_cbd_km'], df['pollution_index'])

print("📊 Spearman Correlation: Distance vs Pollution")
print(f"Correlation coefficient: {spearman_corr:.3f}")
print(f"P-value: {spearman_p:.4f}")

# Compare with Pearson
pearson_dist_poll, _ = stats.pearsonr(df['distance_cbd_km'], df['pollution_index'])
print(f"\n🔍 Comparison:")
print(f"Spearman: {spearman_corr:.3f}")
print(f"Pearson: {pearson_dist_poll:.3f}")
print(f"\n💡 Spearman captures the non-linear relationship better!")

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(df['distance_cbd_km'], df['pollution_index'], alpha=0.5, s=30, color='green')
plt.xlabel('Distance from CBD (km)', fontsize=12)
plt.ylabel('Pollution Index', fontsize=12)
plt.title(f'Distance vs Pollution (Non-linear)\nSpearman ρ = {spearman_corr:.3f}', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Kendall's Tau (Ordinal Data)

In [ ]:
# Kendall's Tau: Also rank-based, better for small samples and ordinal data
kendall_corr, kendall_p = kendalltau(df['school_rating'], df['price'])

print("📊 Kendall's Tau: School Rating vs Price")
print(f"Correlation coefficient: {kendall_corr:.3f}")
print(f"P-value: {kendall_p:.4f}")

# Comparison table
comparison_df = pd.DataFrame({
    'Method': ['Pearson', 'Spearman', 'Kendall'],
    'Measures': ['Linear relationship', 'Monotonic relationship', 'Ordinal association'],
    'Data Type': ['Continuous', 'Continuous/Ordinal', 'Ordinal'],
    'Outlier Sensitive': ['Yes', 'No', 'No'],
    'Best For': ['Normal distributions', 'Non-linear monotonic', 'Small samples']
})

print("\n📋 Correlation Methods Comparison:")
print(comparison_df.to_string(index=False))

## 4. Correlation Matrix

### 4.1 Calculate Full Correlation Matrix

In [ ]:
# Pearson correlation matrix
corr_matrix = df.corr(method='pearson')

print("🔗 Correlation Matrix (Pearson):")
print(corr_matrix.round(3))

### 4.2 Visualize with Heatmap

In [ ]:
# Heatmap with annotations
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1)
plt.title('Correlation Heatmap - Housing Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 How to read:")
print("   🔴 Red = Strong positive correlation")
print("   🔵 Blue = Strong negative correlation")
print("   ⚪ White = No correlation")

### 4.3 Correlation with Target Variable

In [ ]:
# Focus on correlations with price (target)
price_corrs = corr_matrix['price'].sort_values(ascending=False)

print("🎯 Features Correlated with Price:")
print(price_corrs)

# Visualize
plt.figure(figsize=(10, 6))
price_corrs[1:].plot(kind='barh', color=['green' if x > 0 else 'red' for x in price_corrs[1:]])
plt.xlabel('Correlation with Price', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance by Correlation', fontsize=14, fontweight='bold')
plt.axvline(0, color='black', linewidth=1)
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# Insights
print("\n📊 Top Positive Correlations:")
print(price_corrs[1:4])
print("\n📊 Top Negative Correlations:")
print(price_corrs[-3:])

## 5. Multicollinearity Detection

**Multicollinearity** = When features are highly correlated with each other

**Why it's a problem:**
- 🔴 Makes model coefficients unstable
- 🔴 Reduces interpretability
- 🔴 Can lead to overfitting

### 5.1 Find Highly Correlated Feature Pairs

In [ ]:
# Find pairs with |correlation| > 0.7
threshold = 0.7

# Get upper triangle of correlation matrix (avoid duplicates)
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find correlations above threshold
high_corr_pairs = []
for column in upper_tri.columns:
    for index in upper_tri.index:
        corr_value = upper_tri.loc[index, column]
        if abs(corr_value) > threshold:
            high_corr_pairs.append({
                'Feature 1': index,
                'Feature 2': column,
                'Correlation': corr_value
            })

if len(high_corr_pairs) > 0:
    print(f"⚠️ Highly Correlated Pairs (|r| > {threshold}):")
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values('Correlation', ascending=False, key=abs)
    print(high_corr_df.to_string(index=False))
    print("\n💡 Consider removing one feature from each pair")
else:
    print(f"✅ No feature pairs with |correlation| > {threshold}")

### 5.2 Variance Inflation Factor (VIF)

In [ ]:
# VIF measures how much variance is inflated due to multicollinearity
# VIF = 1: No correlation
# VIF = 1-5: Moderate correlation (acceptable)
# VIF = 5-10: High correlation (problematic)
# VIF > 10: Severe multicollinearity (remove feature)

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Select features (exclude target)
X = df.drop('price', axis=1)

# Calculate VIF
vif_data = pd.DataFrame()
vif_data['Feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("📊 Variance Inflation Factor (VIF):")
print(vif_data.to_string(index=False))

# Flag problematic features
high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\n⚠️ Features with VIF > 10 (consider removing):")
    print(high_vif['Feature'].tolist())
else:
    print("\n✅ No severe multicollinearity detected")

## 6. Advanced Visualizations

### 6.1 Clustered Correlation Heatmap

In [ ]:
# Cluster similar features together
plt.figure(figsize=(12, 10))
sns.clustermap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
               linewidths=1, figsize=(12, 10), cbar_kws={"shrink": 0.8})
plt.suptitle('Clustered Correlation Heatmap', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Features that cluster together are similar/correlated")

### 6.2 Correlation Network Graph

In [ ]:
# Show only strong correlations as network
import networkx as nx

# Create network graph
G = nx.Graph()

# Add edges for correlations > 0.5
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.5:
            G.add_edge(corr_matrix.columns[i], corr_matrix.columns[j],
                      weight=abs(corr_val), corr=corr_val)

# Draw network
plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G, k=0.5, iterations=50)

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=2000, node_color='lightblue',
                       alpha=0.7, edgecolors='black')

# Draw edges (thickness = correlation strength)
edges = G.edges()
weights = [G[u][v]['weight'] * 3 for u, v in edges]
colors = ['red' if G[u][v]['corr'] > 0 else 'blue' for u, v in edges]
nx.draw_networkx_edges(G, pos, width=weights, alpha=0.5, edge_color=colors)

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

plt.title('Correlation Network (|r| > 0.5)\nRed = Positive, Blue = Negative',
          fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## 7. Correlation Pitfalls ⚠️

### 7.1 Anscombe's Quartet - Same Correlation, Different Data!

In [ ]:
# Load Anscombe's quartet
anscombe = sns.load_dataset('anscombe')

# Plot all four datasets
g = sns.FacetGrid(anscombe, col='dataset', col_wrap=2, height=4)
g.map(plt.scatter, 'x', 'y', s=50, alpha=0.7)
g.map(sns.regplot, 'x', 'y', scatter=False, color='red')

# Add correlation to each subplot
for ax, (name, data) in zip(g.axes.flat, anscombe.groupby('dataset')):
    corr = data['x'].corr(data['y'])
    ax.set_title(f'Dataset {name}\nCorrelation: {corr:.3f}', fontweight='bold')
    ax.grid(alpha=0.3)

plt.suptitle("Anscombe's Quartet: Always Visualize Your Data!", y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("🎯 Key Lesson: All 4 datasets have nearly identical correlations (~0.816)")
print("   but completely different patterns! ALWAYS VISUALIZE!")

### 7.2 Outliers Can Distort Correlation

In [ ]:
# Create data with outlier
np.random.seed(123)
x_clean = np.random.normal(50, 10, 100)
y_clean = np.random.normal(50, 10, 100)

# Add one outlier
x_outlier = np.append(x_clean, 100)
y_outlier = np.append(y_clean, 100)

corr_clean = np.corrcoef(x_clean, y_clean)[0, 1]
corr_outlier = np.corrcoef(x_outlier, y_outlier)[0, 1]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(x_clean, y_clean, alpha=0.6, s=50)
axes[0].set_title(f'Without Outlier\nCorrelation: {corr_clean:.3f}', fontsize=12, fontweight='bold')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].grid(alpha=0.3)

axes[1].scatter(x_clean, y_clean, alpha=0.6, s=50, label='Normal points')
axes[1].scatter([100], [100], color='red', s=200, marker='*', label='Outlier', edgecolors='black', linewidths=2)
axes[1].set_title(f'With Outlier\nCorrelation: {corr_outlier:.3f}', fontsize=12, fontweight='bold')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Y')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"⚠️ Correlation changed from {corr_clean:.3f} to {corr_outlier:.3f} due to ONE outlier!")
print("💡 Solution: Use Spearman correlation (rank-based, robust to outliers)")

## 8. Practical Workflow

In [ ]:
def correlation_analysis(dataframe, target_col, threshold=0.7):
    """
    Complete correlation analysis workflow
    """
    print("="*60)
    print("📊 CORRELATION ANALYSIS REPORT")
    print("="*60)
    
    # 1. Calculate correlation matrix
    corr_matrix = dataframe.corr()
    
    # 2. Correlations with target
    target_corrs = corr_matrix[target_col].sort_values(ascending=False)
    print(f"\n🎯 Top 5 Features Correlated with {target_col}:")
    print(target_corrs[1:6])
    
    # 3. Multicollinearity check
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_features = [column for column in upper_tri.columns if any(abs(upper_tri[column]) > threshold)]
    
    print(f"\n⚠️ Features with High Multicollinearity (|r| > {threshold}):")
    if len(high_corr_features) > 0:
        print(high_corr_features)
    else:
        print("None found")
    
    # 4. Recommendations
    print("\n📋 Recommendations:")
    print(f"✅ Keep features: {target_corrs[1:4].index.tolist()}")
    if len(high_corr_features) > 0:
        print(f"⚠️ Check multicollinearity: {high_corr_features}")
    print(f"❌ Consider removing: {target_corrs[-3:].index.tolist()} (weak correlation)")
    
    print("\n" + "="*60)
    
    return corr_matrix

# Run analysis
corr_matrix = correlation_analysis(df, 'price', threshold=0.7)

## 9. Your Turn! 💪

**Exercise**: Using the housing dataset:
1. Calculate Spearman correlation for all variables
2. Compare Pearson vs Spearman for `age_years` vs `price`
3. Calculate VIF for all features
4. Identify which feature you would remove first and why

In [ ]:
# Your code here

---

## Key Takeaways 🎯

1. **Pearson**: Linear relationships, sensitive to outliers
2. **Spearman**: Monotonic relationships, robust to outliers
3. **Kendall**: Ordinal data, small samples
4. **Always visualize**: Same correlation ≠ same pattern (Anscombe's Quartet)
5. **Check multicollinearity**: VIF > 10 = problematic
6. **Correlation ≠ Causation**: Strong correlation doesn't imply cause-effect

**Next**: Learn systematic outlier detection techniques! 🔍